<a href="https://colab.research.google.com/github/aaquibmomin2003/LearnMachineLearning/blob/main/Multi_Batch_Training_Loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

# =====================================================================
# 1. CREATE A MASSIVE SIMULATED DATASET
# =====================================================================
# 100 rows of houses, each has 4 features
all_inputs = torch.randn(100, 4)
# 100 target prices matching those houses
all_targets = torch.randn(100, 1) * 100 + 300

# Glue them together into a PyTorch Dataset container
dataset = TensorDataset(all_inputs, all_targets)

# Create the manager to serve data in mini-batches of 20
data_loader = DataLoader(dataset, batch_size=20, shuffle=True)

# =====================================================================
# 2. SETUP THE MODEL AND OPTIMIZER
# =====================================================================
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden_layer = nn.Linear(4, 8)
        self.output_layer = nn.Linear(8, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.output_layer(self.relu(self.hidden_layer(x)))

model = SimpleMLP().to(device)
loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.05)

# =====================================================================
# 3. THE MULTI-BATCH TRAINING LOOP
# =====================================================================
print("--- STARTING MULTI-BATCH TRAINING ---")

for epoch in range(1, 6): # Run for 5 complete epochs
    epoch_loss = 0.0

    # The DataLoader automatically splits our 100 rows into 5 batches of 20!
    for batch_num, (batch_inputs, batch_targets) in enumerate(data_loader, 1):
        # Move the current mini-batch over to your active T4 GPU hardware
        batch_inputs = batch_inputs.to(device)
        batch_targets = batch_targets.to(device)

        # Standard training steps:
        optimizer.zero_grad()
        predictions = model(batch_inputs)
        loss = loss_function(predictions, batch_targets)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    # Calculate average error across all 5 batches
    avg_loss = epoch_loss / len(data_loader)
    print(f"Epoch {epoch} Completed | Average Loss Score: {avg_loss:.4f}")

--- STARTING MULTI-BATCH TRAINING ---
Epoch 1 Completed | Average Loss Score: 95656.5938
Epoch 2 Completed | Average Loss Score: 94516.3328
Epoch 3 Completed | Average Loss Score: 92777.5031
Epoch 4 Completed | Average Loss Score: 90375.6219
Epoch 5 Completed | Average Loss Score: 86921.7406
